# Day28 Hybrid GPU Notebook (Colab)
Notebook này chạy **vLLM + Embedding service** trên Colab GPU và xuất URL để nối về local stack.

## 1) Cài dependencies

In [20]:
!pip -q install vllm fastapi uvicorn pyngrok sentence-transformers requests

## 2) Cấu hình ngrok token

In [ ]:
from pyngrok import ngrok
NGROK_TOKEN = "3DeQj25jmJtjuEeRKzPsM9mqZjh_4mUjDQM9ok8zxhJtEQSLF"
ngrok.set_auth_token(NGROK_TOKEN)
print('ngrok token configured')

ngrok token configured


## 3) Chạy vLLM server (OpenAI-compatible)

In [22]:
import subprocess, threading, time
import requests # Added for health check

VLLM_PORT = 8001
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

def run_vllm():
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_NAME,
        "--host", "0.0.0.0",
        "--port", str(VLLM_PORT),
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.7"
    ])

print('Starting vLLM server in background...')
threading.Thread(target=run_vllm, daemon=True).start()

# Health check loop for vLLM
max_retries = 18 # 18 * 10 seconds = 180 seconds
for i in range(max_retries):
    try:
        # vLLM's OpenAI API server has a /health endpoint
        response = requests.get(f"http://localhost:{VLLM_PORT}/health", timeout=5)
        if response.status_code == 200:
            print(f'vLLM server is healthy after {i*10} seconds.')
            break
    except requests.exceptions.ConnectionError:
        pass # Server not yet ready
    print(f'Waiting for vLLM server... ({i*10}s elapsed)')
    time.sleep(10)
else:
    print('vLLM server did not become healthy within the timeout.')
    # Optionally raise an error or exit if critical


Starting vLLM server in background...
Waiting for vLLM server... (0s elapsed)
Waiting for vLLM server... (10s elapsed)
Waiting for vLLM server... (20s elapsed)
Waiting for vLLM server... (30s elapsed)
Waiting for vLLM server... (40s elapsed)
Waiting for vLLM server... (50s elapsed)
Waiting for vLLM server... (60s elapsed)
Waiting for vLLM server... (70s elapsed)
Waiting for vLLM server... (80s elapsed)
vLLM server is healthy after 90 seconds.


## 4) Expose vLLM qua ngrok

In [23]:
vllm_tunnel = ngrok.connect(8001, "http")
VLLM_NGROK_URL = vllm_tunnel.public_url
print('VLLM_NGROK_URL=', VLLM_NGROK_URL)

VLLM_NGROK_URL= https://roundish-gauging-uniquely.ngrok-free.dev


## 5) Chạy Embedding service

In [24]:
import sys
from unittest import mock
import requests # Added for health check
import time # Added for sleep in health check

# Mock torchcodec.decoders to prevent import errors if not strictly needed
sys.modules['torchcodec.decoders'] = mock.MagicMock()

from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn, threading

app = FastAPI()
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

@app.post('/embed')
def embed(data: dict):
    texts = data.get('texts', [])
    vectors = embed_model.encode(texts).tolist()
    return {'embeddings': vectors}

def run_embed():
    uvicorn.run(app, host='0.0.0.0', port=8002)

EMBED_PORT = 8002 # Define this for clarity in the health check
print(f'Starting embedding service on {EMBED_PORT} in background...')
threading.Thread(target=run_embed, daemon=True).start()

# Health check loop for embedding service
max_retries = 12 # 12 * 5 seconds = 60 seconds (embedding models are usually faster to load)
for i in range(max_retries):
    try:
        # A simple GET on the root might work, or try a POST to /embed with dummy data
        # For FastAPI, a GET on '/' usually works for a basic health check
        response = requests.get(f"http://localhost:{EMBED_PORT}/", timeout=5)
        if response.status_code == 200:
            print(f'Embedding service is healthy after {i*5} seconds.')
            break
    except requests.exceptions.ConnectionError:
        pass # Server not yet ready
    print(f'Waiting for embedding service... ({i*5}s elapsed)')
    time.sleep(5)
else:
    print('Embedding service did not become healthy within the timeout.')
    # Optionally raise an error or exit if critical


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Starting embedding service on 8002 in background...


INFO:     Started server process [1560]
INFO:     Waiting for application startup.


INFO:     127.0.0.1:47850 - "GET / HTTP/1.1" 404 Not Found


INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8002): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Waiting for embedding service... (0s elapsed)
INFO:     127.0.0.1:56304 - "GET / HTTP/1.1" 404 Not Found
Waiting for embedding service... (5s elapsed)
INFO:     127.0.0.1:56308 - "GET / HTTP/1.1" 404 Not Found
Waiting for embedding service... (10s elapsed)
INFO:     127.0.0.1:50844 - "GET / HTTP/1.1" 404 Not Found
Waiting for embedding service... (15s elapsed)
INFO:     127.0.0.1:50848 - "GET / HTTP/1.1" 404 Not Found
Waiting for embedding service... (20s elapsed)
INFO:     127.0.0.1:45622 - "GET / HTTP/1.1" 404 Not Found
Waiting for embedding service... (25s elapsed)
INFO:     127.0.0.1:45624 - "GET / HTTP/1.1" 404 Not Found
Waiting for embedding service... (30s elapsed)
INFO:     127.0.0.1:58782 - "GET / HTTP/1.1" 404 Not Found
Waiting for embedding service... (35s elapsed)
INFO:     127.0.0.1:58796 - "GET / HTTP/1.1" 404 Not Found
Waiting for embedding service... (40s elapsed)
INFO:     127.0.0.1:53154 - "GET / HTTP/1.1" 404 Not Found
Waiting for embedding service... (45s elapsed)
I

## 6) Expose Embedding service qua ngrok

In [25]:
embed_tunnel = ngrok.connect(8002, "http")
EMBED_NGROK_URL = embed_tunnel.public_url
print('EMBED_NGROK_URL=', EMBED_NGROK_URL)

EMBED_NGROK_URL= https://roundish-gauging-uniquely.ngrok-free.dev


## 7) Test nhanh 2 service

In [26]:
import requests

r1 = requests.get(f"{VLLM_NGROK_URL}/health", timeout=60)
print('vLLM health:', r1.status_code, r1.text[:200])

r2 = requests.post(f"{EMBED_NGROK_URL}/embed", json={"texts": ["hello world"]}, timeout=60)
print('embed test:', r2.status_code, len(r2.json().get('embeddings', [[]])[0]))

INFO:     34.143.221.161:0 - "GET /health HTTP/1.1" 404 Not Found
vLLM health: 404 {"detail":"Not Found"}
embed test: 404 0


## 8) Copy vào local `.env`
Dán 2 URL vừa in ra:

```env
VLLM_NGROK_URL=https://xxxx.ngrok-free.app
EMBED_NGROK_URL=https://yyyy.ngrok-free.app
```